# Dara Kyzmet — агент: проверка запросами

Ноутбук гоняет агента поддержки (`POST /api/v1/agent/ask`). Агент работает в режиме
**function-calling**: LLM сам выбирает и вызывает инструменты. **Требуется поднятый vLLM**
(Qwen2.5-VL с tool-calling) — без него `/agent/ask` вернёт 503.

**Перед запуском:**
```bash
bash run_vllm.sh            # отдельный терминал, ждём загрузки модели
docker compose up --build   # db + qdrant + api + web
```
Дождись в логах `api` строки `Каталог проиндексирован в Qdrant: 5 товаров` (поиск товара).
Демо-логины: `store@dara.kz` / `dist@dara.kz`, пароль `demo12345`.

In [ ]:
import httpx  # из зависимостей backend (uv); либо: pip install httpx

BASE_URL = "http://localhost:8080"

client = httpx.Client(base_url=BASE_URL, timeout=60.0)

# health: модель, к которой подключён api (vLLM должен быть поднят)
print(client.get("/health").json())

In [ ]:
def login(email: str, password: str = "demo12345") -> str:
    r = client.post("/api/v1/auth/login", json={"email": email, "password": password})
    r.raise_for_status()
    return r.json()["access_token"]


def ask(token: str, message: str, image_b64: str | None = None) -> dict:
    """Один запрос к агенту: печатает вопрос/ответ и использованные инструменты."""
    body = {"message": message}
    if image_b64:
        body["image_b64"] = image_b64
    r = client.post(
        "/api/v1/agent/ask",
        headers={"Authorization": f"Bearer {token}"},
        json=body,
    )
    r.raise_for_status()
    data = r.json()
    print(f"❓ {message}")
    print(f"💬 {data['answer']}")
    print(f"🛠  used_tools={data.get('used_tools')}\n")
    return data


store_token = login("store@dara.kz")
print("store token:", store_token[:24], "...")

## 1. Stock-агент (остатки)
Демо засеян: по 5 шт. каждого товара (Молоко, Кефир, Сметана, Творог, Масло).

In [ ]:
ask(store_token, "покажи остатки")              # все позиции (get_stock без фильтра)
ask(store_token, "сколько Молоко на складе?")   # фильтр по названию
ask(store_token, "что заканчивается?")          # low_stock: всё < 10 -> вернёт все 5

## 2. Analytics-агент (расхождения, тренды)
Расхождений в свежем демо нет — пока акт не сформирован, отчёт пустой.

In [ ]:
ask(store_token, "покажи расхождения")          # discrepancy_report
ask(store_token, "какие тренды продаж?")        # top_products (по qty_ordered в заявках)
ask(store_token, "топ товаров")                 # top_products

## 3. Product-агент (поиск по каталогу через Qdrant)
Бьёт в Qdrant с фильтром по `org_id`. Требует, чтобы индексация при старте прошла.

In [ ]:
ask(store_token, "найди товар кефир в каталоге")      # search_by_text
ask(store_token, "сопоставь позицию: Сметана 20%")    # search_by_text
ask(store_token, "что за товар по артикулу масло")    # search_by_text

## 4. Fallback (вне известных интентов)

In [ ]:
ask(store_token, "привет, что ты умеешь?")

## 5. Изоляция тенантов
Каталог принадлежит магазину. Тот же product-запрос под поставщиком не должен ничего найти.

In [ ]:
dist_token = login("dist@dara.kz")
ask(dist_token, "найди товар кефир в каталоге")   # ожидаем: совпадений нет (чужой тенант)
ask(dist_token, "покажи остатки")                 # у поставщика своего стока нет

## 6. (Опционально) Поиск товара по фото
Поиск по фото (CLIP image→text по каталогу). Укажи путь к изображению.

In [ ]:
import base64
from pathlib import Path

IMG_PATH = ""  # напр. "/home/nurda/photo.jpg"; оставь пустым, чтобы пропустить

if IMG_PATH and Path(IMG_PATH).exists():
    img_b64 = base64.b64encode(Path(IMG_PATH).read_bytes()).decode()
    ask(store_token, "что это за товар?", image_b64=img_b64)
else:
    print("IMG_PATH не задан — пропускаем фото-поиск.")

## 7. Order-агент: создание заявки (предложение → подтверждение)
Агент НЕ создаёт заявку сам: сначала предлагает черновик (`data.draft`),
и только при повторном запросе с `confirm_draft` создаёт заявку в статусе `new`.
Количества «как обычно» берутся из прошлых заявок (демо: молоко 20, кефир 30).

In [ ]:
# Шаг 1 — ПРЕДЛОЖЕНИЕ (read-only, ничего не пишет)
r = ask(store_token, "Закажи молоко, кефир и сметану как обычно на завтра")
draft = r["data"].get("draft")
print("📋 draft:", draft)

In [ ]:
# Шаг 2 — ПОДТВЕРЖДЕНИЕ (создаётся черновик заявки, статус new)
if draft:
    resp = client.post(
        "/api/v1/agent/ask",
        headers={"Authorization": f"Bearer {store_token}"},
        json={"message": "Подтверждаю создание черновика", "confirm_draft": draft},
    )
    resp.raise_for_status()
    out = resp.json()
    print("✅", out["answer"])
    print("created_order:", out["data"].get("created_order"))
else:
    print("Черновик не предложен — проверь сид/каталог.")

In [ ]:
# Проверяем, что заявка реально появилась в списке (статус new)
orders = client.get("/api/v1/orders",
                    headers={"Authorization": f"Bearer {store_token}"}).json()
print("заявок всего:", len(orders))
for o in orders[:5]:
    items = ", ".join(f"{it['name']} x{it['qty_ordered']:g}" for it in o["items"])
    print(" ", o["id"][:8], o["status"], "|", items)

### Явные количества и поставщик по имени
Можно указать количества прямо в тексте; если поставщиков несколько — назвать его.

In [ ]:
ask(store_token, "закажи молоко 20, кефир 30 шт")     # qty из текста
ask(store_token, "оформи заявку: масло и творог")      # qty по прошлым заявкам или 1

## 8. Новая аналитика (NLU: исход / поставщики / статус / траты)
Запрос обрабатывает function-calling агент: LLM сам выбирает инструменты.
На свежих демо-данных расхождений и подтверждённых накладных ещё нет, поэтому
рейтинг поставщиков и траты будут пустыми, пока не пройдёшь приёмку с расхождениями.

In [ ]:
# Что на исходе + у кого дозаказать (stock < порога + поставщик)
ask(store_token, "что на исходе?")
ask(store_token, "что нужно заказать на неделю?")

In [ ]:
# Рейтинг поставщиков по браку/недостачам (Discrepancy -> Acceptance -> Order)
ask(store_token, "у кого больше всего брака и недостач?")

In [ ]:
# Статус заявок по поставщику
ask(store_token, "в каком статусе заявки к Молпром?")

In [ ]:
# Сколько потратили за период (подтверждённые накладные)
ask(store_token, "сколько ушло Молпром в этом месяце?")
ask(store_token, "сколько потратили за прошлый месяц?")